# Data Preprocessing Overview

## 1. Goal & Approach
Convert messy, nested data from 7 retail websites into a clean, flat table for PostgreSQL and Business Intelligence tools (PowerBI/Tableau).
- **Standardize Keys:** Group different Vietnamese words (e.g. `ổ cứng`, `lưu trữ`) into standard keys.
- **Extract Info:** Use Regex to pull exact numbers and hardware categories (RAM, CPU, GPU, social metrics).
- **Handle Missing Data:** Keep missing specs as `Null/NaN` to ensure charts are accurate (no fake data). Unrecognized categories are set to `"Other"`.

## 2. Flat Schema (28 Columns)
- **Meta:** `id`, `source`, `url`, `crawl_date`
- **General:** `brand`, `name`, `category` *(Office / Gaming / Creator)*, `is_ai_laptop` *(True / False)*, `is_available`
- **Price:** `pricing_current_price`, `pricing_price_segment` *(Budget <15tr, Mainstream 15-25tr, Mid-high 25-45tr, Premium >45tr)*
- **CPU / GPU:** `spec_cpu_raw`, `spec_cpu_brand`, `spec_cpu_family`, `spec_gpu_type` *(Dedicated / Integrated / Unified)*, `spec_gpu_model`
- **RAM / Storage:** `spec_ram_capacity_gb` *(number only)*, `spec_ram_type`, `spec_storage_capacity_gb`, `spec_storage_type`
- **Hardware:** `spec_display_size_inches`, `spec_display_resolution` *(e.g., FHD, 2K, 4K)*, `spec_display_refresh_rate_hz`, `spec_design_weight_kg`, `spec_os_family`
- **Social:** `social_avg_rating`, `social_review_count`, `social_satisfied_count`

## Data Loading

In [1]:
import json
import os
import re
import pandas as pd
import unicodedata

# Define the data directory
DATA_DIR = "data"

def load_json_files(data_dir):
    all_data = []
    files_processed = 0
    # Collect all json files starting with 'data_'
    for filename in os.listdir(data_dir):
        if filename.startswith('data_') and filename.endswith('.json'):
            file_path = os.path.join(data_dir, filename)
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    if isinstance(data, list):
                        all_data.extend(data)
                    else:
                        all_data.append(data)
                files_processed += 1
            except Exception as e:
                print(f"Error {filename}: {e}")
    print(f"Total files: {files_processed}")
    print(f"Total records: {len(all_data)}")
    return all_data

raw_records = load_json_files(DATA_DIR)

Total files: 7
Total records: 3732


## Text Cleaners & Regex 

In [2]:
def remove_accents(input_str):
    if input_str is None:
        return ""
    # Strip accents and transform strings to lowercase
    s1 = unicodedata.normalize('NFD', input_str)
    s2 = "".join([c for c in s1 if not unicodedata.combining(c)])
    # Special handle for letter d
    s2 = s2.replace('đ', 'd').replace('Đ', 'D')
    return s2.lower().strip()

def normalize_key(key):
    return remove_accents(key)

In [3]:
def extract_ram_capacity(ram_string):
    if not ram_string: return None
    # Step 1: Detect explicit standard representations like "16 GB"
    match = re.search(r'(\d+)\s*(?:gb|g\b)', str(ram_string), re.IGNORECASE)
    if match:
        return int(match.group(1))
    
    # Step 2: Fallback logic for standalone values e.g. "16"
    match2 = re.search(r'\b(4|8|12|16|24|32|48|64|96|128)\b', str(ram_string))
    if match2:
        return int(match2.group(1))
        
    return None

def extract_ram_type(ram_string):
    if not ram_string: return None
    ram_string = str(ram_string).upper()
    if 'LPDDR5X' in ram_string: return 'LPDDR5X'
    if 'LPDDR5' in ram_string: return 'LPDDR5'
    if 'DDR5' in ram_string: return 'DDR5'
    if 'LPDDR4X' in ram_string: return 'LPDDR4X'
    if 'LPDDR4' in ram_string: return 'LPDDR4'
    if 'DDR4' in ram_string: return 'DDR4'
    if 'UNIFIED' in ram_string or 'M-SERIES' in ram_string: return 'Unified'
    return None

def extract_storage_capacity(storage_string):
    if not storage_string: return None
    tb_match = re.search(r'(\d+)\s*(?:tb|tera)', str(storage_string), re.IGNORECASE) 
    if tb_match:
        return int(tb_match.group(1)) * 1024

    gb_match = re.search(r'(\d+)\s*(?:gb|g\b)', str(storage_string), re.IGNORECASE)  
    if gb_match:
        return int(gb_match.group(1))
    
    # Step 3: Handle raw digits indicative of storage scales
    st2 = re.search(r'\b(128|256|512)\b', str(storage_string))
    if st2: return int(st2.group(1))
    
    return None

def extract_storage_type(storage_string):
    if not storage_string: return None
    storage_string = str(storage_string).upper()
    if 'SSD' in storage_string: return 'SSD'
    if 'HDD' in storage_string: return 'HDD'
    if 'EMMC' in storage_string: return 'eMMC'
    return 'SSD'

def extract_screen_size(screen_string):
    if not screen_string: return None
    match = re.search(r'(\d{2}(?:\.\d+)?)\s*(?:\"|inch|\'\')', str(screen_string), re.IGNORECASE)
    if match:
        return float(match.group(1))
    match_start = re.search(r'^(\d{2}(?:\.\d+)?)', str(screen_string))
    if match_start:
        return float(match_start.group(1))
    return None

def extract_screen_hz(screen_string):
    if not screen_string: return None
    match = re.search(r'(\d+)\s*(?:hz)', str(screen_string), re.IGNORECASE)
    if match:
        return int(match.group(1))
    return 60

def extract_screen_resolution(screen_string):
    if not screen_string: return None
    screen_string = str(screen_string).upper()
    if '4K' in screen_string or '3840' in screen_string: return '4K'
    if '3K' in screen_string or '2880' in screen_string: return '3K'
    if '2.8K' in screen_string or '2880X1800' in screen_string or '2880 X 1800' in screen_string: return '2.8K'
    if '2.5K' in screen_string or 'WQXGA' in screen_string or '2560' in screen_string: return '2.5K'       
    if '2K' in screen_string: return '2K'
    if 'FHD+' in screen_string or '1920 X 1200' in screen_string or '1920X1200' in screen_string or 'WUXGA' in screen_string: return 'FHD+'
    if 'FHD' in screen_string or 'FULL HD' in screen_string or '1920 X 1080' in screen_string or '1920X1080' in screen_string: return 'FHD'
    if 'HD' in screen_string: return 'HD'
    return None

def extract_gpu_type(gpu_string):
    if not gpu_string: return "Integrated"
    gpu_str_lower = str(gpu_string).lower()
    dedicated_keywords = ['rtx', 'gtx', 'rx', 'radeon pro', 'nvdia', 'nvidia', 'geforce', 'arc', 'mx', 'quadro']
    unified_keywords = ['apple', 'unified']

    for kw in dedicated_keywords:
        if kw in gpu_str_lower: return "Dedicated"
    for kw in unified_keywords:
        if kw in gpu_str_lower: return "Unified"
    return "Integrated"

def extract_gpu_model(gpu_raw):
    if not gpu_raw: return "Other"
    
    # Remove unreadable trademark chars blocking Regex matchers
    g_str = str(gpu_raw).upper().replace('™', '').replace('®', '')
    
    # 1. NVIDIA RTX / Quadro
    rtx_match = re.search(r'(RTX\s*(?:PRO\s*)?(?:A)?\d{3,4}(?:\s*TI|\s*SUPER|\s*ADA)?)', g_str)
    if rtx_match: return "NVIDIA " + re.sub(r'\s+', ' ', rtx_match.group(1))
    
    gtx_match = re.search(r'(GTX\s*\d{4}(?:\s*TI)?)', g_str)
    if gtx_match: return "NVIDIA " + re.sub(r'\s+', ' ', gtx_match.group(1))

    quadro_match = re.search(r'(QUADRO\s*[A-Z]?\d{3,4})', g_str)
    if quadro_match: return "NVIDIA " + re.sub(r'\s+', ' ', quadro_match.group(1))
    
    mx_match = re.search(r'(MX\s*\d{3}A?)', g_str)
    if mx_match: return "NVIDIA GeForce " + re.sub(r'\s+', ' ', mx_match.group(1))

    # 2. AMD
    rx_match = re.search(r'(RX\s*\d{4}(?:\s*M|\s*S)?)', g_str)
    if rx_match: return "AMD Radeon " + re.sub(r'\s+', ' ', rx_match.group(1))

    amd_m = re.search(r'(RADEON\s*\d{3}M)', g_str)
    if amd_m: return "AMD " + re.sub(r'\s+', ' ', amd_m.group(1))
    
    if 'RADEON' in g_str: return "AMD Radeon Graphics"

    # 3. QUALCOMM
    if 'ADRENO' in g_str or 'SNAPDRAGON' in g_str: return "Qualcomm Adreno"

    # 4. INTEL
    if 'ARC' in g_str: return "Intel Arc"
    if 'IRIS' in g_str or 'X\\' in g_str or 'XE' in g_str: return "Intel Iris Xe"
    if 'UHD' in g_str: return "Intel UHD"
    if 'HD GRAPHICS' in g_str: return "Intel HD Graphics"

    # 5. APPLE 
    if 'APPLE' in g_str or 'NHÂN GPU' in g_str or 'CORE GPU' in g_str or 'M1' in g_str or 'M2' in g_str or 'M3' in g_str or 'M4' in g_str: 
        return "Apple GPU"

    # 6. Fallback Integrated
    if 'INTEL GRAPHIC' in g_str or 'INTEGRATED GRAPHIC' in g_str or 'ONBOARD' in g_str or 'ON BOAD' in g_str or 'SHARED' in g_str: 
        return "Integrated Graphics"
    
    return "Other"

def extract_os_family(os_raw):
    if not os_raw: return "Windows" # Default assumption mapped to Windows if unknown
    os_str = str(os_raw).lower()
    if 'windows 11' in os_str or 'win 11' in os_str: return "Windows 11"
    if 'windows 10' in os_str or 'win 10' in os_str: return "Windows 10"
    if 'mac' in os_str: return "macOS"
    if 'dos' in os_str or 'linux' in os_str or 'ubuntu' in os_str: return "DOS/Linux"
    if 'windows' in os_str or 'win' in os_str: return "Windows"
    return "Windows"

def extract_weight_kg(weight_raw):
    if not weight_raw: return None
    match = re.search(r'(\d+[\.,]\d+|\d+)\s*(?:kg)', str(weight_raw), re.IGNORECASE)
    if match:
        val = match.group(1).replace(',', '.')
        try:
            return float(val)
        except ValueError:
            return None
    
    m2 = re.search(r'(?:nặng|lượng|weight)[\s\:]*(\d+[\.,]\d+|\d+)', str(weight_raw).lower())
    if m2:
        val = m2.group(1).replace(',', '.')
        try:
            return float(val)
        except ValueError:
            pass
    return None

def extract_cpu_family(cpu_raw):
    if not cpu_raw: return "Other"
    cpu_str = str(cpu_raw).lower()

    apple_match = re.search(r'(m[1-5](?:\s*pro|\s*max)?)', cpu_str)
    if apple_match: return f"Apple {apple_match.group(1).title()}"

    ultra_match = re.search(r'ultra\s*(\d)', cpu_str)
    if ultra_match: return f"Core Ultra {ultra_match.group(1)}"

    core_i_match = re.search(r'core\s*(?:i)?(\d)', cpu_str)
    if core_i_match: return f"Core i{core_i_match.group(1)}"
    if 'pentium' in cpu_str: return "Intel Pentium"
    if 'celeron' in cpu_str: return "Intel Celeron"
    if 'n100' in cpu_str or 'n200' in cpu_str: return "Intel N-series"

    ryzen_ai_match = re.search(r'ryzen\s*ai\s*(\d)', cpu_str)
    if ryzen_ai_match: return f"Ryzen AI {ryzen_ai_match.group(1)}"

    ryzen_match = re.search(r'ryzen\s*(?:z?\d)', cpu_str)
    if ryzen_match:
        match_str = ryzen_match.group(0).title()
        return match_str.replace("Z1", "Z1")

    if 'snapdragon' in cpu_str: return "Snapdragon"
    return "Other"

def get_price_segment(price):
    if price is None or price == 0: return None
    if price < 15000000: return "Budget"
    if price <= 25000000: return "Mainstream"
    if price <= 45000000: return "Mid-high"
    return "Premium"

def get_category(name, gpu_type, screen_hz, ram_gb):
    name_lower = str(name).lower() if name else ""

    gaming_keywords = ['gaming', 'nitro', 'tuf', 'rog ', 'legion', 'alienware', 'predator', 'loq', 'ideapad gaming', 'victus', 'cyborg', 'katana', 'bravo', 'stealth', 'raider', 'omen', 'tuf dash']
    if any(kw in name_lower for kw in gaming_keywords): return "Gaming"
    if gpu_type == "Dedicated" and screen_hz and screen_hz >= 120: return "Gaming"

    creator_keywords = ['creator', 'studio', 'proart', 'conceptd', 'zenbook pro']
    if any(kw in name_lower for kw in creator_keywords): return "Creator"       
    if ram_gb and ram_gb >= 32 and gpu_type == "Dedicated": return "Creator"    

    office_keywords = ['vivobook', 'zenbook', 'expertbook', 'swift', 'aspire', 'ideapad', 'thinkpad', 'thinkbook', 'macbook', 'elitebook', 'hp 14', 'hp 15', 'hp 240', 'hp 245', 'hp 250']    
    if any(kw in name_lower for kw in office_keywords): return "Office"

    return "Office"

def check_ai_chip(cpu_raw, name):
    combine_str = f"{cpu_raw or ''} {name or ''}".lower()
    ai_keywords = ['npu', 'ai boost', 'ultra', 'ryzen ai', 'snapdragon', 'copilot', 'xdna']
    return any(kw in combine_str for kw in ai_keywords)

def parse_satisfied_count(text):
    if not text: return 0
    match_k = re.search(r'([\d\.,]+)\s*(k|K|triệu)?\s*(khách|lượt)', str(text).lower())
    if match_k:
        num_str = match_k.group(1).replace(',', '.')
        try:
            num = float(num_str)
        except ValueError:
            return 0
        unit = match_k.group(2)
        if unit == 'k': return int(num * 1000)
        elif unit == 'triệu': return int(num * 1000000)
        return int(num)

    match_digit = re.search(r'^\s*([\d\.,]+)', str(text))
    if match_digit:
        num_str = match_digit.group(1).replace(',', '.')
        try:
            return int(float(num_str))
        except ValueError:
            pass
    return 0

## Transformer Class

In [4]:
class LaptopTransformer:
    def __init__(self):
        # Dictionary mappings consolidating wild keys into unified targets
        self.key_mapping = {
            'cpu': ['cpu', 'vi xu ly'], # Captures CPU type, frequencies
            'ram': ['ram', 'bo nho trong'], # Captures capacities, frequencies, buses
            'storage': ['o cung', 'rom', 'storage', 'ssd', 'hdd', 'luu tru'],
            'vga': ['vga', 'card', 'do hoa', 'gpu'], # Handles discrete, integrated variants
            'screen': ['man hinh', 'hien thi', 'display', 'do phan giai', 'tan so', 'tam nen', 'screen'], # Screen scale, panels, resolutions
            'os': ['os', 'he dieu hanh', 'operating system'],
            'weight': ['weight', 'khoi luong', 'trong luong', 'kich thuoc', 'nang', 'trong']
        }

    def _find_spec(self, raw_specs, spec_type):
        """Comb specs dict using keyword logic, compounding valid hits to prevent structural truncation"""
        target_keys = self.key_mapping.get(spec_type, [])
        found_values = []
        for k, v in raw_specs.items():
            norm_k = normalize_key(k)
            # Match
            if any(tk in norm_k for tk in target_keys):
                found_values.append(str(v).strip())
        return " | ".join(found_values)

    def process(self, raw_data):
        # Eject visibly corrupted or irrelevant items (Bags, mice, accessories)
        name = raw_data.get('product', {}).get('name', '')
        if not name or "balo" in name.lower() or "chuột" in name.lower() or "túi" in name.lower() or "apple care" in name.lower():
            return None

        current_price = raw_data.get('pricing', {}).get('current_price', 0)     
        is_available = raw_data.get('product', {}).get('is_available', True)    

        # Sanity check: Exclude missing product pricings
        if current_price == 0:
            return None

        # Validate dict length for parameter viability
        specs_dict = raw_data.get('specs_raw', {})
        if not specs_dict:
             return None

        # Manufacture (Brand) string mapping
        brand_match = re.search(r'(Asus|Acer|Dell|HP|Lenovo|MSI|Apple|Macbook|Gigabyte|LG|Microsoft)', name, re.IGNORECASE)
        if brand_match:
            b_name = brand_match.group(1).title()
            brand = "Apple" if b_name == "Macbook" else ("HP" if b_name == "Hp" else ("MSI" if b_name == "Msi" else ("LG" if b_name == "Lg" else b_name)))      
        else:
            brand = "Other"

        # Root feature raw nodes extraction
        raw_cpu = self._find_spec(specs_dict, 'cpu')
        raw_ram = self._find_spec(specs_dict, 'ram')
        raw_storage = self._find_spec(specs_dict, 'storage')
        raw_vga = self._find_spec(specs_dict, 'vga')
        raw_screen = self._find_spec(specs_dict, 'screen')
        raw_os = self._find_spec(specs_dict, 'os')
        raw_weight = self._find_spec(specs_dict, 'weight')

        # CPU
        cpu_str_lower = raw_cpu.lower()
        if "intel" in cpu_str_lower or "core" in cpu_str_lower:
            cpu_brand = "Intel"
        elif "amd" in cpu_str_lower or "ryzen" in cpu_str_lower:
            cpu_brand = "AMD"
        elif "apple" in cpu_str_lower or "m1" in cpu_str_lower or "m2" in cpu_str_lower or "m3" in cpu_str_lower or "m4" in name.lower() or brand == "Apple":   
            cpu_brand = "Apple"
        elif "snapdragon" in cpu_str_lower:
            cpu_brand = "Qualcomm"
        else:
            cpu_brand = "Other"
            
        cpu_family = extract_cpu_family(raw_cpu)
        if cpu_family == "Other" and brand == "Apple":
            # Mac-specific fallback: Infer SoC specs securely from product titles
            m_match = re.search(r'\b(m[1-5](?:\s*pro|\s*max)?)\b', name.lower())
            if m_match:
                cpu_family = f"Apple {m_match.group(1).title()}"

        is_ai = check_ai_chip(raw_cpu + str(specs_dict), name)

        # GPU
        gpu_type = extract_gpu_type(raw_vga)
        gpu_model = extract_gpu_model(raw_vga)
        if brand == "Apple" and gpu_model == "Other":
            gpu_model = "Apple GPU"

        # RAM
        ram_gb = extract_ram_capacity(raw_ram)
        ram_type = extract_ram_type(raw_ram)

        # Storage
        storage_gb = extract_storage_capacity(raw_storage)
        storage_type = extract_storage_type(raw_storage)

        # Screen
        screen_size = extract_screen_size(raw_screen)
        screen_hz = extract_screen_hz(raw_screen)
        screen_res = extract_screen_resolution(raw_screen)
        
        # OS & Weight
        os_family = extract_os_family(raw_os)
        if brand == "Apple": os_family = "macOS" # Force fix Apple OS
        
        weight_kg = extract_weight_kg(raw_weight)

        # Categories
        cat = get_category(name, gpu_type, screen_hz, ram_gb)

        # Build JSON Output
        master_json = {
            "id": raw_data.get("id"),
            "source": raw_data.get("metadata", {}).get("source"),
            "url": raw_data.get("metadata", {}).get("url"),
            "crawl_date": raw_data.get("metadata", {}).get("crawl_date"),
            
            # Contextual Classification logic
            "brand": brand,
            "name": name,
            "category": cat,
            "is_ai_laptop": is_ai,
            "is_available": is_available,
            
            # Pricing
            "pricing_current_price": current_price,
            "pricing_price_segment": get_price_segment(current_price),

            # CPU
            "spec_cpu_raw": raw_cpu,
            "spec_cpu_brand": cpu_brand,
            "spec_cpu_family": cpu_family,

            # GPU
            "spec_gpu_type": gpu_type,
            "spec_gpu_model": gpu_model,
            
            # RAM
            "spec_ram_capacity_gb": ram_gb,
            "spec_ram_type": ram_type,

            # Storage
            "spec_storage_capacity_gb": storage_gb,
            "spec_storage_type": storage_type,

            # Display
            "spec_display_size_inches": screen_size,
            "spec_display_resolution": screen_res,
            "spec_display_refresh_rate_hz": screen_hz,

            # Other
            "spec_os_family": os_family,
            "spec_design_weight_kg": weight_kg,
            
            # Social
            "social_avg_rating": raw_data.get("social", {}).get("avg_rating", 0.0),      
            "social_review_count": raw_data.get("social", {}).get("review_count", 0),        
            "social_satisfied_count": parse_satisfied_count(raw_data.get("social", {}).get("satisfied_info", ""))
        }

        return master_json

## ETL Data Processor

In [5]:
import pandas as pd
import json
import os

transformer = LaptopTransformer()
cleaned_dataset = []

for raw in raw_records:
    try:
        processed = transformer.process(raw)
        if processed is not None:
            cleaned_dataset.append(processed)
    except Exception as e:
        print(f"Exception on node ID: {raw.get('id')} - {e}")

print(f"Processed records: {len(cleaned_dataset)}")

# Serialize finalized dataset array out to JSON
output_file = os.path.join(DATA_DIR, "transformed_data.json")
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(cleaned_dataset, f, ensure_ascii=False, indent=4)

Processed records: 2798
